In [43]:
import pandas as pd

from tzlocal import get_localzone
from datetime import datetime
import plotly.graph_objects as go
import warnings
warnings.simplefilter('ignore')
warnings.filterwarnings('ignore')
from tqdm import tqdm
tqdm.pandas()

import train_model as tm
from train_model import ModelFunc
import data_processing as dp
from data_loader import load_data_at_start_date, load_data_by_period
from features import FeatureEngineering

import optuna
from optuna import Trial, visualization
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [44]:
dir = 'crypto_data'
symbol = 'BTC-USD'
period= -(datetime.now() - datetime(2018, 1, 1)).days
load_data_at_start_date([symbol], period, '1d', dir)
data = dp.get_data(dir, symbol, compress=False)


Start load data, tickers ['BTC-USD'], interval: 1d, start date: -2610
date: 2018-01-01 15:27:53.119143


[*********************100%***********************]  1 of 1 completed

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2611 entries, 2018-01-01 to 2025-02-23
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   (BTC-USD, Open)    2611 non-null   float64
 1   (BTC-USD, High)    2611 non-null   float64
 2   (BTC-USD, Low)     2611 non-null   float64
 3   (BTC-USD, Close)   2611 non-null   float64
 4   (BTC-USD, Volume)  2611 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 122.4 KB
Download data completed


### Features

In [45]:
fe_params = {
    'emaf': 30,
    'emam': 150,
    'emas': 200,
    'rsi': 14,
    'macd': [12, 26, 9],
 }
indicators = ['emaf', 'emam', 'emas', 'rsi', 'macd']

fe = FeatureEngineering(fe_params)

In [46]:
lag_periods = 3
features_to_trend = ['Open', 'High', 'Low', 'Close', 'Volume']
data_with_features = fe.clear_invalid_targets(fe.add_target(fe.enrich_with_indicators(data), lag_periods))
data_with_trend, new_trend_features = fe.create_trend_features(data_with_features, features_to_trend, lag_periods) 
features = new_trend_features + indicators
data_with_trend = data_with_trend[features + ['Target']]
# display(data_with_trend)

### Split data

In [47]:
split_params = {
    'max_train_size': 0.5,
    'test_size': 0.4,
}
train_data, val_data, test_data = tm.train_valid_test_split_data(data_with_trend, split_params)

X_train, y_train = tm.split_by_features_and_target_variables(train_data, features)
X_val, y_val = tm.split_by_features_and_target_variables(val_data, features)
X_test, y_test = tm.split_by_features_and_target_variables(test_data, features)

print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")

Train size: 1203, Val size: 721, Test size: 482


In [48]:
X_train_scaled, X_val_scaled, X_test_scaled = tm.normalize_StandardScaler(X_train, X_val, X_test)
    
model = tm.fit_models([ModelFunc.CATBOOST_CLASS], X_train_scaled, y_train, X_val_scaled, y_val)[0]
models_proba = tm.predict_models([model], X_train_scaled, X_val_scaled, X_test_scaled)[0]

y_train_pred_prob = models_proba['train']
y_val_pred_prob = models_proba['val']
y_test_pred_prob = models_proba['test']

train_roc_auc = tm.roc_auc_score_metric(y_train, y_train_pred_prob)
val_roc_auc = tm.roc_auc_score_metric(y_val, y_val_pred_prob)
test_roc_auc = tm.roc_auc_score_metric(y_test, y_test_pred_prob)

In [49]:
print("=== Train sample metrics ===")
print(f"ROC AUC: {train_roc_auc:.4f}")
tm.calculate_metrics_table(y_train, y_train_pred_prob)


=== Train sample metrics ===
ROC AUC: 0.7786


,Cutoff,Precision,Recall,Accuracy,F1-Score
0,50.0,65.804935,86.956522,68.827930,74.916388
1,60.0,100.000000,5.900621,49.625935,11.143695
2,70.0,0.000000,0.000000,46.467165,0.000000
3,80.0,0.000000,0.000000,46.467165,0.000000


In [50]:
print("=== Validation sample metrics ===")
print(f"ROC AUC: {val_roc_auc:.4f}")
tm.calculate_metrics_table(y_val, y_val_pred_prob)

=== Validation sample metrics ===
ROC AUC: 0.5384


,Cutoff,Precision,Recall,Accuracy,F1-Score
0,50.0,50.15873,89.014085,51.040222,64.162437
1,60.0,100.00000,1.408451,51.456311,2.777778
2,70.0,0.00000,0.000000,50.762829,0.000000
3,80.0,0.00000,0.000000,50.762829,0.000000


In [51]:
print("=== Test sample metrics ===")
print(f"ROC AUC: {test_roc_auc:.4f}")
tm.calculate_metrics_table(y_test, y_test_pred_prob)

=== Test sample metrics ===
ROC AUC: 0.5005


,Cutoff,Precision,Recall,Accuracy,F1-Score
0,50.0,55.988858,75.0,53.319502,64.114833
1,60.0,0.000000,0.0,44.398340,0.000000
2,70.0,0.000000,0.0,44.398340,0.000000
3,80.0,0.000000,0.0,44.398340,0.000000


### Hyperparameter tuning with GridSearchCV

In [52]:
params = {
    'grid_params': {
        'depth': [4, 6, 8, 10],                      # Разные значения глубины дерева
        'learning_rate': [0.01, 0.05, 0.1, 0.2],     # Разные темпы обучения
        'n_estimators': [100, 200, 500],             # Разные количества деревьев
        'l2_leaf_reg': [1, 3, 5, 7],                 # Разные коэффициенты L2-регуляризации
        #takes long time to run
        # 'bagging_temperature': [0, 0.3, 0.6, 1],   # Разные температуры бэггинга
        # 'rsm': [0.5, 0.8, 1.0],                    # Разные доли признаков (RSM)
        # 'subsample': [0.5, 0.8, 1.0],              # Разные доли выборки для каждого дерева
    },
    'params': {
        'scoring': 'roc_auc', # accuracy
        'cv': 5,                        # Количество фолдов для кросс-валидации
        'n_jobs': -1,
        'random_state': 42,
        'verbose': 0,
        'early_stopping_rounds': 50, # Activates Iter overfitting detector with od_wait parameter 
        'task_type': 'GPU',
    }
}

grid_search = tm.catboot_classifier_model_grid_search(X_train_scaled, y_train, X_val_scaled, y_val, params)

# Получаем лучшую модель и параметры
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print("Best params:", best_params)
print("Best ROC AUC Score:", best_score)


Best params: {'depth': 8, 'l2_leaf_reg': 1, 'learning_rate': 0.2, 'n_estimators': 100}
Best ROC AUC Score: 0.47809581117402705


### Optuna: hyperparameter optimization

In [53]:
study = tm.optuna_study_CatBoostClassifier(X_train_scaled, y_train, X_val_scaled, y_val, n_trials=5)

best_params = study.best_params
best_score = study.best_value

print("best params:", best_params)
print("best ROC AUC Score:", best_score)

best params: {'iterations': 328, 'depth': 3, 'learning_rate': 0.09271679036450096, 'l2_leaf_reg': 0.012059335779291941, 'bagging_temperature': 0.7604862907967752, 'rsm': 0.9619414833138648, 'subsample': 0.5795244433277513}
best ROC AUC Score: 0.4619840737183867


In [54]:
#shows the relative importances of hyperparameters.
tm.optuna_plot_param_importances(study)

In [55]:
#plots the empirical distribution function of the objective.
tm.optuna_plot_edf(study)

### Using best params

In [56]:
best_params['random_state'] = 42
best_params['verbose'] = 0

model = tm.model_fit_with_eval(ModelFunc.CATBOOST_CLASS, X_train_scaled, y_train, (X_val_scaled, y_val), best_params)
models_proba = tm.predict_models([model], X_train_scaled, X_val_scaled, X_test_scaled)[0]

y_train_pred_prob = models_proba['train']
y_val_pred_prob = models_proba['val']
y_test_pred_prob = models_proba['test']

train_roc_auc = tm.roc_auc_score_metric(y_train, y_train_pred_prob)
val_roc_auc = tm.roc_auc_score_metric(y_val, y_val_pred_prob)
test_roc_auc = tm.roc_auc_score_metric(y_test, y_test_pred_prob)

In [57]:
print("=== Train sample metrics ===")
print(f"ROC AUC: {train_roc_auc:.4f}")
tm.calculate_metrics_table(y_train, y_train_pred_prob)

=== Train sample metrics ===
ROC AUC: 0.5368


,Cutoff,Precision,Recall,Accuracy,F1-Score
0,50.0,54.664342,97.360248,55.361596,70.01675
1,60.0,0.000000,0.000000,46.467165,0.00000
2,70.0,0.000000,0.000000,46.467165,0.00000
3,80.0,0.000000,0.000000,46.467165,0.00000


In [58]:
print("=== Validation sample metrics ===")
print(f"ROC AUC: {val_roc_auc:.4f}")
tm.calculate_metrics_table(y_val, y_val_pred_prob)

=== Validation sample metrics ===
ROC AUC: 0.4994


,Cutoff,Precision,Recall,Accuracy,F1-Score
0,50.0,49.281609,96.619718,49.375867,65.27117
1,60.0,0.000000,0.000000,50.762829,0.00000
2,70.0,0.000000,0.000000,50.762829,0.00000
3,80.0,0.000000,0.000000,50.762829,0.00000


In [59]:
print("=== Test sample metrics ===")
print(f"ROC AUC: {test_roc_auc:.4f}")
tm.calculate_metrics_table(y_test, y_test_pred_prob)

=== Test sample metrics ===
ROC AUC: 0.5161


,Cutoff,Precision,Recall,Accuracy,F1-Score
0,50.0,55.532359,99.253731,55.394191,71.218206
1,60.0,0.000000,0.000000,44.398340,0.000000
2,70.0,0.000000,0.000000,44.398340,0.000000
3,80.0,0.000000,0.000000,44.398340,0.000000
